In [40]:
import re
import mgrs

import pandas as pd
import geopandas as gpd

from zipfile import ZipFile
from datetime import datetime
from shapely.geometry import Point
from ipyleaflet import Map, GeoData, basemaps, LayersControl

In [43]:
def read_sm_zip(path: str) -> list:
    """
    Takes a path to a zip file containing stridsmeldinger.
    Returns a list of raw stridsmelding strings.
    """
    stridsmeldinger = []
    with ZipFile(path) as sm_zip:
        for file_name in sm_zip.namelist():
            with sm_zip.open(file_name) as sm:
                stridsmeldinger.append(sm.read().decode('utf-8'))
    return stridsmeldinger

def find_substring_between(start: str, stop: str, stridsmelding):
    start = re.escape(start)
    end   = re.escape(stop)
    result = re.search('%s(.*)%s' % (start, end), stridsmelding).group(1)
    return result.strip()

def parse_sm_date(date_str: str) -> datetime:
    return datetime.strptime(date_str, '%d%H%MZ%b%y')

def from_mgrs(coordinate: str) -> Point:
    m = mgrs.MGRS()
    lat, lon = m.toLatLon(coordinate)
    return Point(lon,lat)

def parse_stridsmelding(stridsmelding):
    date_str = find_substring_between("DTG", "\n", stridsmelding)
    sm_datetime = parse_sm_date(date_str)
    
    sm_from = find_substring_between("FRA:", "\n", stridsmelding)
    sm_to = find_substring_between("TIL:", "\n", stridsmelding)
    sm_message = find_substring_between("\n\n", "\n", stridsmelding)

    pos_mgrs = find_substring_between("posisjon", "DTG", sm_message)
    pos_point = from_mgrs(pos_mgrs)
    
    sign = find_substring_between("---", "---", stridsmelding)
    
    return {
        'datetime': sm_datetime,
        'from': sm_from,
        'to': sm_to,
        'message': sm_message,
        'sign': sign,
        'geometry': pos_point
    }


In [44]:
stridsmeldinger = read_sm_zip('data/stridsmelding.zip')

data = []
for sm in stridsmeldinger:
    data.append(parse_stridsmelding(sm))

df = pd.DataFrame(data)

In [38]:
df.apply(lambda x: from_mgrs(x.pos_mgrs), axis=1)

0     1 M113 har TATT EN TEKNISK HVIL i posisjon 32V...
1     3 M113 har KJØRT SEG FAST i posisjon 32VMM7965...
2     5 BV206 har GÅTT I BIUVAKK i posisjon 32VMN523...
3     2 BV206 har KJØRT SEG FAST i posisjon 32VMN968...
4     3 IVECO LMV har GÅTT I STILLING i posisjon 32V...
                            ...                        
95    1 M113 har GÅTT I STILLING i posisjon 32VMN696...
96    5 MB FELTVOGN har GÅTT I STILLING i posisjon 3...
97    4 BV206 har TATT FYR i posisjon 32VMN933222119...
98    1 MB FELTVOGN har KJØRT SEG FAST i posisjon 32...
99    1 DINGO 2 har GÅTT I BIUVAKK i posisjon 32VMM9...
Name: melding, Length: 100, dtype: object